# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### Reflection:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# load the hotel bookings data
url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv'
df = pd.read_csv(url)
print(df.shape)
print(df.dtypes.value_counts())
print(df.isna().sum()[df.isna().sum() > 0])
print(df['hotel'].value_counts())

# drop columns that reveal the outcome, the mostly empty company column, and hotel (one value only)
df = df.drop(columns=['reservation_status', 'reservation_status_date', 'assigned_room_type', 'company', 'hotel'])

X = df.drop(columns='is_canceled')
y = df['is_canceled']
print(y.value_counts(normalize=True))

# split 70/30 before any preprocessing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# fill missing country and agent with placeholder values
X_train = X_train.fillna({'country': 'Unknown', 'agent': 0})
X_test = X_test.fillna({'country': 'Unknown', 'agent': 0})

# agent is an ID number, so treat it as a category
X_train['agent'] = X_train['agent'].astype(int).astype(str)
X_test['agent'] = X_test['agent'].astype(int).astype(str)

# one-hot encode categories, then match the test columns to the training columns
X_train = pd.get_dummies(X_train, dtype=int)
X_test = pd.get_dummies(X_test, dtype=int).reindex(columns=X_train.columns, fill_value=0)
print(X_train.shape, X_test.shape)

(9404, 32)
int64      17
object     12
float64     3
Name: count, dtype: int64
country     266
agent      2335
company    8393
dtype: int64
hotel
Resort Hotel    9404
Name: count, dtype: int64
is_canceled
0    0.751701
1    0.248299
Name: proportion, dtype: float64
(6582, 228) (2822, 228)


### ✍️ Your Response: 🔧
1. The raw dataset has 9,404 rows and 32 columns. After I dropped five columns and one-hot encoded the categories, the training set has 6,582 rows and the test set has 2,822 rows, each with 228 feature columns.

2. The raw data has 17 integer columns, 3 float columns, and 12 text (object) columns. Numeric features include lead time, nights stayed, guest counts, and average daily rate. Categorical features include arrival month, meal, country, market segment, room type, deposit type, and customer type. I treated agent as categorical because it is an ID number.

3. I dropped reservation_status and reservation_status_date because they record the final outcome, and assigned_room_type because it is set at check-in. I dropped company (8,393 missing) and hotel, which is Resort Hotel for all 9,404 rows. After a stratified 70/30 split, I filled missing country (266) with "Unknown" and missing agent (2,335) with 0, then one-hot encoded the categories with matching train and test columns.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.


In [7]:
import time
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# train Naive Bayes and time the fit
nb = GaussianNB()
start = time.time()
nb.fit(X_train, y_train)
nb_time = time.time() - start

# predict on the test set and evaluate
nb_pred = nb.predict(X_test)
nb_acc = accuracy_score(y_test, nb_pred)
print(classification_report(y_test, nb_pred))
print(confusion_matrix(y_test, nb_pred))
print('Training time (seconds):', round(nb_time, 2))

              precision    recall  f1-score   support

           0       0.98      0.42      0.59      2121
           1       0.36      0.98      0.52       701

    accuracy                           0.56      2822
   macro avg       0.67      0.70      0.56      2822
weighted avg       0.83      0.56      0.57      2822

[[ 887 1234]
 [  14  687]]
Training time (seconds): 0.06


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?

### ✍️ Your Response: 🔧
1. The model performs poorly. Its test accuracy is 0.56, below the 0.75 share of bookings that were not canceled. It catches 687 of 701 cancellations (recall 0.98), but only 0.36 of its cancel flags are correct, and it flags 1,234 kept bookings. Since only about a quarter of bookings cancel, F1 for the canceled class (0.52) is a better measure than accuracy.

2. Naive Bayes trained in 0.12 seconds and missed only 14 cancellations, so it could work as a first screen for low-cost actions, such as a reminder email to every flagged guest. Its 1,234 false alarms make it a poor tool for operational decisions like staffing or overbooking.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# fit the scaler on the training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# train a linear SVM and time the fit
svm = SVC(kernel='linear')
start = time.time()
svm.fit(X_train_scaled, y_train)
svm_time = time.time() - start

# predict on the test set and evaluate
svm_pred = svm.predict(X_test_scaled)
svm_acc = accuracy_score(y_test, svm_pred)
print(classification_report(y_test, svm_pred))
print(confusion_matrix(y_test, svm_pred))
print('Training time (seconds):', round(svm_time, 2))

              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2121
           1       0.82      0.76      0.79       701

    accuracy                           0.90      2822
   macro avg       0.87      0.85      0.86      2822
weighted avg       0.90      0.90      0.90      2822

[[2005  116]
 [ 167  534]]
Training time (seconds): 3.63


### Reflection:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?

### ✍️ Your Response: 🔧
1. The linear SVM performs well, with test accuracy of 0.90. For the canceled class, precision is 0.82, recall is 0.76, and F1 is 0.79. It missed 167 of the 701 cancellations and wrongly flagged 116 kept bookings. F1 for the canceled class is still the best measure, because accuracy rewards the larger non-canceled class.

2. SVM fits decisions where a false alarm costs money, such as overbooking rooms or cutting staff for expected cancellations. It wrongly flagged 116 kept bookings, compared with 1,234 for Naive Bayes, and it raised F1 for cancellations from 0.52 to 0.79. That makes its flags far more useful for planning than those of the simpler model.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

In [9]:
from sklearn.neural_network import MLPClassifier

# two hidden layers; early_stopping holds out 10% of the training data as a validation set
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, validation_fraction=0.1,
                    max_iter=500, random_state=42)
start = time.time()
mlp.fit(X_train_scaled, y_train)
mlp_time = time.time() - start
print('Best validation accuracy:', round(mlp.best_validation_score_, 3))

# predict on the test set and evaluate
mlp_pred = mlp.predict(X_test_scaled)
mlp_acc = accuracy_score(y_test, mlp_pred)
print(classification_report(y_test, mlp_pred))
print(confusion_matrix(y_test, mlp_pred))
print('Training time (seconds):', round(mlp_time, 2))

Best validation accuracy: 0.926
              precision    recall  f1-score   support

           0       0.93      0.95      0.94      2121
           1       0.83      0.79      0.81       701

    accuracy                           0.91      2822
   macro avg       0.88      0.87      0.87      2822
weighted avg       0.91      0.91      0.91      2822

[[2008  113]
 [ 149  552]]
Training time (seconds): 1.82


### Reflection:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?

### ✍️ Your Response: 🔧
1. The neural network had the highest test accuracy at 0.91, compared with 0.90 for the SVM and 0.56 for Naive Bayes. For cancellations it reached 0.83 precision, 0.79 recall, and 0.81 F1, slightly above the SVM on each. Its best validation accuracy, from held-out training data, was 0.926. It missed 149 cancellations.

2. The business would be comfortable using it for some decisions but not all. The results show the model predicts well, but it cannot tell a manager why it flagged a booking. That is acceptable for planning totals, such as how many rooms to overbook next week. For actions aimed at one guest, such as requiring a deposit, the hotel would need a reason it can explain.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

In [10]:
# compare test accuracy and training time for all three models
results = pd.DataFrame({
    'Model': ['Naive Bayes', 'SVM (linear)', 'Neural Network'],
    'Accuracy': [nb_acc, svm_acc, mlp_acc],
    'Training time (s)': [nb_time, svm_time, mlp_time]
})
print(results.round(3))

            Model  Accuracy  Training time (s)
0     Naive Bayes     0.558              0.058
1    SVM (linear)     0.900              3.633
2  Neural Network     0.907              1.823


### Reflection:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?

### ✍️ Your Response: 🔧
1. The neural network had the best accuracy (0.907), with the SVM close behind (0.900) and Naive Bayes far back (0.558). Naive Bayes trained fastest (0.118 seconds), then the neural network (4.926) and the SVM (13.667). Naive Bayes is the easiest to interpret and needed no scaling. The linear SVM is easier to explain than the neural network.

2. Yes, I would recommend the neural network for a trial deployment. It had the best accuracy and the best F1 for cancellations (0.81), and it trained in 4.926 seconds, faster than the SVM at 13.667. The SVM is only slightly behind and easier to explain. The hotel should test the model on new bookings before it drives staffing or overbooking decisions.

## 6. Final Business Recommendation

### Reflection:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?

2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I recommend the neural network model for predicting which bookings will cancel. In testing, it identified 552 of 701 cancellations and was correct on 91 percent of all bookings. Managers can use its flags to plan staffing, room inventory, and overbooking. The model has two limits: it cannot explain why it flagged a booking, and it missed 149 cancellations. The data covers only the Resort Hotel, so it says nothing about city hotel guests. Adding City Hotel bookings would allow a city and resort comparison and could improve future results.

2. My learning outcomes are: (1) "I will be able to build a Python workflow that cleans and consistently categorizes messy client transaction data, so a diligence team can flag non-recurring items and quality-of-earnings adjustments without re-sorting by hand." (2) "I will be able to evaluate AI-generated categorizations against the source financials by treating each output as a hypothesis to verify rather than an answer to trust, so I can frame which discrepancies actually matter before they reach a quality-of-earnings conclusion." This assignment practiced both. For the first, I filled missing country and agent values and encoded categories the same way in the training and test data. For the second, I judged each model against a baseline instead of trusting accuracy. Naive Bayes scored 0.56, below the 0.75 share of kept bookings, so its output did not hold up.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [11]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 325643 bytes to assignment_12_bayes_svm_neural.html
